# QADD v4.2.0 — Scientific finalization and freeze preparation

This notebook does **not** decode audio or recompute QADD features. It verifies the completed cohort candidate, preserves the five numerical feature columns exactly, applies the final scientific roles, corrects the eligible hum-winner summary, creates the completed validation records and feature passports, and prepares an atomic-freeze candidate.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT_OVERRIDE = None  # installer inserts the local project root
RUN_PACKAGE_TESTS = True
RUN_FINALIZATION = True

SCIENTIFIC_REVIEW_DECISION = "PENDING"  # installer sets ACCEPT_QADD_V420
SCIENTIFIC_REVIEWER = "Nevena Musikic"
SCIENTIFIC_REVIEW_RATIONALE = (
    "Post-cohort analytical-validation review completed. Five observable QADD measurements "
    "are retained under feature-specific claim boundaries; no family scalar or standalone gate is approved."
)

PUBLISH_AND_FREEZE = False  # the notebook never freezes directly

def locate_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "MAIN outputs").exists()
            and (candidate / "src").exists()
            and (candidate / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the project root")

PROJECT_ROOT = (
    Path(PROJECT_ROOT_OVERRIDE).resolve()
    if PROJECT_ROOT_OVERRIDE
    else locate_project_root(Path.cwd().resolve())
)
SOURCE_ROOT = PROJECT_ROOT / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates" / "additive_interference" / "qadd-v4.2.0-candidate"
FINAL_ROOT = PROJECT_ROOT / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates" / "additive_interference" / "qadd-v4.2.0-final-candidate"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks/02_feature_extraction" / "02_QADD"
SRC_REVIEWED = PROJECT_ROOT / "src"
if str(SRC_REVIEWED) not in sys.path:
    sys.path.insert(0, str(SRC_REVIEWED))

from paper1_qc_reviewed.qadd_v420_final import (
    ACCEPTANCE_TOKEN,
    ANALYSIS_FEATURES,
    finalize_candidate,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source cohort candidate: {SOURCE_ROOT}")
print(f"Final candidate: {FINAL_ROOT}")
print(f"Review decision: {SCIENTIFIC_REVIEW_DECISION}")
print(f"Publish and freeze in notebook: {PUBLISH_AND_FREEZE}")


In [ ]:
if RUN_PACKAGE_TESTS:
    test_paths = [
        PROJECT_ROOT / "tests" / "test_qadd_v420.py",
        PROJECT_ROOT / "tests" / "test_qadd_v420_cohort.py",
        PROJECT_ROOT / "tests" / "test_qadd_v420_final.py",
    ]
    command = [sys.executable, "-m", "pytest", *map(str, test_paths), "-q", "--disable-warnings"]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=PROJECT_ROOT, check=False)
    if completed.returncode:
        raise RuntimeError("QADD reviewed test suite failed")


In [ ]:
source_manifest_path = SOURCE_ROOT / "manifests" / "qadd_v420_cohort_candidate_manifest.json"
source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
required = {
    "cohort_extraction_completed": True,
    "recording_count": 519,
    "participant_count": 224,
    "required_panels_complete": True,
    "freeze_allowed": False,
}
for key, expected in required.items():
    observed = source_manifest.get(key)
    if observed != expected:
        raise ValueError(f"Source manifest mismatch for {key}: {observed!r} != {expected!r}")

for relative in [
    "audit/qadd_v420_extraction_errors.csv",
    "audit/qadd_v420_robustness_errors.csv",
    "audit/qadd_v420_gallery_errors.csv",
]:
    table = pd.read_csv(SOURCE_ROOT / relative)
    if len(table):
        raise ValueError(f"Source error table is not empty: {relative}")

source_features = pd.read_csv(SOURCE_ROOT / "tables" / "qadd_v420_recording_features.csv")
if len(source_features) != 519:
    raise ValueError("Unexpected recording count")
if not all(feature in source_features for feature in ANALYSIS_FEATURES):
    raise ValueError("One or more analysis features are missing")
print("Source cohort candidate verified.")


In [ ]:
provenance_files = [
    NOTEBOOK_DIR / "support/QADD_v420_FINAL_SCIENTIFIC_AUDIT.md",
    NOTEBOOK_DIR / "support/QADD_v420_FINAL_FEATURE_DECISIONS.csv",
    NOTEBOOK_DIR / "support/QADD_Family_Evaluation_Workbook_v1_0.docx",
    NOTEBOOK_DIR / "support/QADD_V4_2_0_FREEZE_CONTRACT.md",
    NOTEBOOK_DIR / "support/QADD_Validation_Checklist_v1_0.csv",
    NOTEBOOK_DIR / "support/QADD_Ten_Domain_Dashboard_v1_0.csv",
]

if RUN_FINALIZATION:
    if FINAL_ROOT.exists():
        archive = FINAL_ROOT.with_name(FINAL_ROOT.name + ".pre_finalization_backup")
        if archive.exists():
            shutil.rmtree(archive)
        FINAL_ROOT.rename(archive)
        print(f"Archived previous final candidate: {archive}")

    manifest = finalize_candidate(
        source_root=SOURCE_ROOT,
        final_root=FINAL_ROOT,
        scientific_review_decision=SCIENTIFIC_REVIEW_DECISION,
        scientific_reviewer=SCIENTIFIC_REVIEWER,
        scientific_review_rationale=SCIENTIFIC_REVIEW_RATIONALE,
        provenance_files=provenance_files,
    )
    print(json.dumps(manifest, indent=2))
else:
    manifest = {}


In [ ]:
if RUN_FINALIZATION:
    final_features = pd.read_csv(FINAL_ROOT / "tables" / "qadd_v420_recording_features.csv")
    source_features = pd.read_csv(SOURCE_ROOT / "tables" / "qadd_v420_recording_features.csv")
    equality = {}
    for feature in ANALYSIS_FEATURES:
        a = pd.to_numeric(source_features[feature], errors="coerce").to_numpy(float)
        b = pd.to_numeric(final_features[feature], errors="coerce").to_numpy(float)
        equality[feature] = bool(np.array_equal(a, b, equal_nan=True))
    equality_table = pd.DataFrame(
        [{"feature": key, "numerically_identical": value} for key, value in equality.items()]
    )
    display(equality_table)
    if not all(equality.values()):
        raise RuntimeError("Numerical equivalence failed")

    hum_summary = pd.read_csv(FINAL_ROOT / "tables" / "qadd_v420_hum_joint_evidence_summary.csv")
    display(hum_summary)
    if not bool(hum_summary.loc[0, "winner_counts_match_eligible"]):
        raise RuntimeError("Eligible hum winner counts do not reconcile")

    dashboard = pd.read_csv(FINAL_ROOT / "validation" / "qadd_v420_ten_domain_dashboard.csv")
    decisions = pd.read_csv(FINAL_ROOT / "validation" / "qadd_v420_g10_feature_decisions.csv")
    figure_index = pd.read_csv(FINAL_ROOT / "figures" / "qadd_v420_standardized_figure_index.csv")
    display(dashboard)
    display(decisions[["feature", "final_decision", "publication_role"]])
    print(f"Standardized figure rows: {len(figure_index)}")


In [ ]:
if RUN_FINALIZATION:
    final_manifest_path = FINAL_ROOT / "manifests" / "qadd_v420_final_candidate_manifest.json"
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))

    if SCIENTIFIC_REVIEW_DECISION == ACCEPTANCE_TOKEN:
        assert final_manifest["freeze_allowed"] is True
        assert final_manifest["freeze_status"] == "ready_for_atomic_freeze"
        assert final_manifest["numerical_equivalence_to_cohort_candidate"] is True
        assert final_manifest["hum_winner_counts_match_eligible"] is True
        print("QADD v4.2.0 FINALIZATION COMPLETE")
        print("Candidate is ready for atomic freeze.")
    else:
        assert final_manifest["freeze_allowed"] is False
        print("QADD v4.2.0 FINALIZATION COMPLETE — FREEZE BLOCKED")
        print(f"Set the exact acceptance token: {ACCEPTANCE_TOKEN}")
